In [18]:
!pip install requests beautifulsoup4

In [19]:
import requests
import json
import time
from bs4 import BeautifulSoup
from collections import Counter

In [20]:
BASE_URL = "https://quotes.toscrape.com"
HEADERS = {
    "User-Agent": "StudentLearningProject/1.0"
}

In [21]:
response = requests.get(
    BASE_URL,
    headers=HEADERS,
    timeout=10)
response.raise_for_status()
print("Website connected successfully!")
print("Status Code:", response.status_code)

Website connected successfully!
Status Code: 200


In [22]:
soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.get_text(strip=True))

Quotes to Scrape


In [23]:
quotes = []
for q in soup.select("div.quote"):
    text = q.select_one("span.text")
    author = q.select_one("small.author")
    quotes.append({
        "text": text.get_text(strip=True) if text else "",
        "author": author.get_text(strip=True) if author else "",
        "tags": [
            tag.get_text(strip=True)
            for tag in q.select("a.tag")
        ]
    })
print("Total quotes extracted:", len(quotes))

Total quotes extracted: 10


In [24]:
def scrape_quotes():
    url = BASE_URL + "/"
    all_quotes = []
    while url:
        try:
            response = requests.get(
                url,
                headers=HEADERS,
                timeout=10
            )
            response.raise_for_status()
            soup = BeautifulSoup(
                response.text,
                "html.parser"
            )
            for q in soup.select("div.quote"):
                text = q.select_one("span.text")
                author = q.select_one("small.author")
                all_quotes.append({
                    "text": text.get_text(strip=True) if text else "",
                    "author": author.get_text(strip=True) if author else "",
                    "tags": [
                        t.get_text(strip=True)
                        for t in q.select("a.tag")
                    ]
                })
            next_page = soup.select_one("li.next a")
            if next_page:
                url = BASE_URL + next_page["href"]
                time.sleep(1)
            else:
                url = None
        except requests.RequestException as error:
            print("Request failed:", error)
            break
    return all_quotes
quotes = scrape_quotes()
print("Total quotes collected:", len(quotes))

Total quotes collected: 100


In [25]:
with open("quotes.json", "w", encoding="utf-8") as file:
    json.dump(
        quotes,
        file,
        indent=2,
        ensure_ascii=False
    )
print("Quotes saved successfully to quotes.json")

Quotes saved successfully to quotes.json


In [26]:
with open("quotes.txt", "w", encoding="utf-8") as file:
    for i, quote in enumerate(quotes, start=1):
        file.write(f"Quote {i}\n")
        file.write(f"Text: {quote['text']}\n")
        file.write(f"Author: {quote['author']}\n")
        file.write(f"Tags: {', '.join(quote['tags'])}\n")
        file.write("-" * 60 + "\n")
print("Quotes saved successfully to quotes.txt")

Quotes saved successfully to quotes.txt


In [27]:
for i, quote in enumerate(quotes[:10], start=1):
    print("Quote:", i)
    print("Text:", quote["text"])
    print("Author:", quote["author"])
    print("Tags:", ", ".join(quote["tags"]))
    print("-" * 50)

Quote: 1
Text: “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
Author: Albert Einstein
Tags: change, deep-thoughts, thinking, world
--------------------------------------------------
Quote: 2
Text: “It is our choices, Harry, that show what we truly are, far more than our abilities.”
Author: J.K. Rowling
Tags: abilities, choices
--------------------------------------------------
Quote: 3
Text: “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
Author: Albert Einstein
Tags: inspirational, life, live, miracle, miracles
--------------------------------------------------
Quote: 4
Text: “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”
Author: Jane Austen
Tags: aliteracy, books, classic, humor
--------------------------------------------------
Quote: 5
Text: “Imperfection is beauty, madness 

In [28]:
author_count = Counter(
    quote["author"] for quote in quotes
)
tag_count = Counter(
    tag
    for quote in quotes
    for tag in quote["tags"]
)
print("PROJECT SUMMARY")
print("------------------------------")
print("Total Quotes:", len(quotes))
if author_count:
    print("Most Quoted Author:", author_count.most_common(1)[0][0])
if tag_count:
    print("Most Common Tag:", tag_count.most_common(1)[0][0])
print("------------------------------")

PROJECT SUMMARY
------------------------------
Total Quotes: 100
Most Quoted Author: Albert Einstein
Most Common Tag: love
------------------------------


In [29]:
print("UNIQUE AUTHORS:", len(author_count))
print("UNIQUE TAGS:", len(tag_count))
print("\nAUTHOR DETAILS:")
for author, count in author_count.most_common():
    print(author, ":", count)
print("\nTOP 10 TAGS:")
for tag, count in tag_count.most_common(10):
    print(tag, ":", count)

UNIQUE AUTHORS: 50
UNIQUE TAGS: 137

AUTHOR DETAILS:
Albert Einstein : 10
J.K. Rowling : 9
Marilyn Monroe : 7
Dr. Seuss : 6
Mark Twain : 6
Jane Austen : 5
C.S. Lewis : 5
Bob Marley : 3
Eleanor Roosevelt : 2
Ralph Waldo Emerson : 2
Mother Teresa : 2
George R.R. Martin : 2
Ernest Hemingway : 2
Charles Bukowski : 2
Suzanne Collins : 2
André Gide : 1
Thomas A. Edison : 1
Steve Martin : 1
Douglas Adams : 1
Elie Wiesel : 1
Friedrich Nietzsche : 1
Allen Saunders : 1
Pablo Neruda : 1
Garrison Keillor : 1
Jim Henson : 1
Charles M. Schulz : 1
William Nicholson : 1
Jorge Luis Borges : 1
George Eliot : 1
Martin Luther King Jr. : 1
James Baldwin : 1
Haruki Murakami : 1
Alexandre Dumas fils : 1
Stephenie Meyer : 1
Helen Keller : 1
George Bernard Shaw : 1
J.R.R. Tolkien : 1
Alfred Tennyson : 1
Terry Pratchett : 1
J.D. Salinger : 1
George Carlin : 1
John Lennon : 1
W.C. Fields : 1
Ayn Rand : 1
Jimi Hendrix : 1
J.M. Barrie : 1
E.E. Cummings : 1
Khaled Hosseini : 1
Harper Lee : 1
Madeleine L'Engle : 1



In [30]:
readme = """
WEB SCRAPER FOR HEADLINES & QUOTES
PROJECT: Python Programming - Project 04
DESCRIPTION:
This project is a web scraping application developed
using Python, Requests and BeautifulSoup.
The application collects quotes, authors and tags
from the Quotes to Scrape practice website.
WEBSITE SOURCE:
https://quotes.toscrape.com/
TOOLS USED:
1. Python
2. Requests
3. BeautifulSoup
4. JSON
FEATURES:
1. Fetches website HTML content.
2. Extracts quotes and authors.
3. Collects tags.
4. Scrapes multiple pages.
5. Saves data in JSON format.
6. Saves readable data in text format.
7. Displays project statistics.
POLITENESS RULES:
1. Uses a timeout for HTTP requests.
2. Waits one second between page requests.
3. Uses a descriptive User-Agent.
4. Uses a practice website intended for scraping.
5. Does not collect private or login-protected data.
OUTPUT FILES:
quotes.json
quotes.txt
CONCLUSION:
This project demonstrates how Python can be used
to collect and organize website information using
Requests and BeautifulSoup.
"""

with open("README.txt", "w", encoding="utf-8") as file:
    file.write(readme)

print("README file created successfully!")

README file created successfully!


In [31]:
import os
print("PROJECT FILES")
print("----------------------")
for filename in os.listdir():
    if filename in [
        "quotes.json",
        "quotes.txt",
        "README.txt"
    ]:
        print(filename)

PROJECT FILES
----------------------
quotes.txt
README.txt
quotes.json


In [32]:
from google.colab import files
files.download("quotes.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
files.download("quotes.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
files.download("README.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
import zipfile
with zipfile.ZipFile(
    "Web_Scraper_Project.zip",
    "w"
) as zip_file:
    zip_file.write("quotes.json")
    zip_file.write("quotes.txt")
    zip_file.write("README.txt")
files.download("Web_Scraper_Project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>